# Bag-of-Words SFT

Run supervised fine-tuning on the synthetic bag-of-words regression task using the shared BoW study config/state.

In [ ]:
from pathlib import Path
import os

import polars as pl
import torch

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    os.chdir(repo_root.parent)
    repo_root = Path.cwd()

from src.experiments.bag_of_words.sft import BagOfWordsSFTConfig

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

## Configure

In [ ]:
config = BagOfWordsSFTConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sft",
    snr=1.0,
    aux_words_ratio=0.0,
    num_words=7,
    num_samples=50_000,
    prompt_length=256,
    word_decay_power=0.0,
    batch_size=64,
    eval_batch_size_multiple=4,
    lr_per_token=2.3e-8,
    backbone_lr_divisor=6.66,
    pad_to_multiple=8,
    train_epochs=10,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset R^2 target: {config.data.rsq:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sft/7-words_snr-1.0_len-256_pow-0.0_ar-0.0
Dataset R^2 target: 0.5000
Backbone lr: 5.658e-05
Head lr:     3.768e-04


## Train

In [ ]:
state.run_training()

sft epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

## Results

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()